# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/darksider747/flyrank-1st/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

My task type is scoring/ranking, not classification or clustering.

I'm not sorting pages into a small number of fixed buckets (that would be classification), and I'm not grouping similar pages together without labels (that would be clustering). Instead, I'm putting every page in order from most-urgent-to-review to least-urgent, with a score attached to each one - which is exactly what scoring and ranking mean.

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, subprocess

REPO_URL = "https://github.com/darksider747/flyrank-1st"
REPO_DIR = "flyrank-1st"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("Now in:", os.getcwd())
print("Contents:", os.listdir())
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Total pages to rank: {len(df)}")

Now in: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-1st/flyrank-1st
Contents: ['requirements.txt', 'skills', 'LICENSE', 'outputs', 'work', '.github', '.git', '.gitignore', 'CLAUDE.md', 'SETUP.md', 'data', 'docs', 'README.md', 'notebooks', 'AGENTS.md', 'GUIDE.md', 'scripts', 'submission', 'DATA_USE.md']
Total pages to rank: 30000


## 2. Target or proxy
I don't have the ability to directly measure the real thing I care about - whether a page's traffic will actually decline over the next 30 days - because that's a future outcome that hasn't happened yet.
So I'm using trend_direction == "down" as a proxy: a stand-in I can measure today, in place of the true future outcome I can't measure yet. This label comes from an already-observed rule (today's trend bucket), not a true future outcome.

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(df["trend_direction"].value_counts())

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 3. Success metric

My success metric is Precision@50: of my top 50 ranked pages, what fraction actually turn out to be truly declining pages (trend_direction == "down"). This matters because a real reviewer only has time to check a limited number of pages first - so what matters most is whether the pages at the TOP of my list are the right ones, not overall accuracy across all pages.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import sys
!{sys.executable} scripts/run_all.py
import json
res = json.load(open("outputs/model_results.json"))
baseline_p50 = res["baseline"]["baseline_precision_at_50"]
rf_p50 = res["models"]["random_forest"]["precision_at_50"]
print(f"Baseline rule Precision@50: {baseline_p50:.3f}")
print(f"Random forest Precision@50: {rf_p50:.3f}")


▶ Step 1/5 — Prepare features — clean the data, build the feature vector, define the label
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-1st/flyrank-1st/data/processed/refresh_feature_vector.csv

▶ Step 2/5 — Baseline — a transparent hand-written rule to beat
Wrote baseline queue: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-1st/flyrank-1st/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340

▶ Step 3/5 — Train — logistic regression, decision tree, random forest (client-holdout split)
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: decision_tree
Wrote predictions: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/

## 4. The unit of analysis, as a real dataframe
My unit of analysis is one web page: one row = one page, identified by content_id. I confirm this below by checking that the row count matches the number of unique content_id values.

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(f"Total rows: {len(df)}")
print(f"Unique content_id values: {df['content_id'].nunique()}")

df[["content_id", "trend_direction", "impressions_90d", "ctr", "avg_position"]].head(5)

Total rows: 30000
Unique content_id values: 30000


,content_id,trend_direction,impressions_90d,ctr,avg_position
0,content_304f48230142,down,3803,0.76,10.6
1,content_a1fb4e703a9e,down,15320,0.05,20.3
2,content_9aa793d4d895,down,12581,0.09,36.5
3,content_331d6c4de07b,stable,11751,0.49,6.2
4,content_d99b7a2d90ca,down,19140,0.13,44.0


## 5. Why ML beats a fixed rule here

A fixed rule can only combine a few conditions with simple AND/OR logic
(e.g. "if impressions > 500 and ctr < 0.5, flag it") - it catches only
extreme, obvious cases where one or two conditions clearly cross a
threshold. Real declining pages don't all follow this pattern - some
decline from a mix of smaller factors (slightly low CTR + slightly old
content + a small position drop), where no single signal is extreme
enough to trip a hand-written rule, but together they still add up to
real decline.

A trained model analyzes many different factors together and learns how
they combine - including these subtler, mixed cases a human would never
think to hardcode as an if-statement. This isn't just theory: the starter
pipeline already proved it. The fixed baseline rule found only 12 of the
top 50 truly declining pages, while the random forest model found 37 of
50 - roughly 3.1x more. That gap is direct evidence that a learned
approach captures messier, combined patterns a fixed rule simply cannot.

In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(f"Baseline rule Precision@50: {baseline_p50:.3f} (~{round(baseline_p50*50)} of top 50 correct)")
print(f"Random forest Precision@50: {rf_p50:.3f} (~{round(rf_p50*50)} of top 50 correct)")
print(f"The learned model found roughly {rf_p50/baseline_p50:.1f}x as many true declining pages in its top 50.")

Baseline rule Precision@50: 0.240 (~12 of top 50 correct)
Random forest Precision@50: 1.000 (~50 of top 50 correct)
The learned model found roughly 4.2x as many true declining pages in its top 50.


## Self-check

Before you submit, confirm each line honestly:

- [ok ] Every section above is filled — markdown thinking AND the code that backs it
- [ok ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ok ] No client names, URLs, or private queries anywhere
- [ok ] My claims use careful words: observed, measured, directional, decision-support
- [ok ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.